# Accelerate Qwen3-8B with Speculative Decoding and Efficient Draft Models

[Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B) is part of the latest [Qwen family](https://qwenlm.github.io/blog/qwen3/), trained with explicit agentic capabilities. It supports tool invocation, multi-step reasoning, and long context, making it well-suited for agent workflows. Integrated with agentic frameworks such as Hugging Face SmolAgents and QwenAgent, it enables a wide range of agentic applications involving tool calling and reasoning

In this notebook we will demonstrate how to speedup Qwen3-8B inference with speculative decoding using OpenVINO GenAI library.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Generation with OpenVINO GenAI](#Generation-with-OpenVINO-GenAI)
- [Accelerated Generation with Speculative Decoding](#Accelerated-Generation-with-Speculative-Decoding)
- [Further Accelerate Qwen3-8B by Draft Layer Pruning ](#Further-Accelerate-Qwen3-8B-by-Draft-Layer-Pruning)
- [Compute Average Speedup Gain of Speculative Decoding with Qwen3-8B model](#Compute-Average-Speedup-Gain-of-Speculative-Decoding-with-Qwen3-8B-model)



## Prerequisites

> Note: we recommend running this notebook in a virtual environment. 

Install required dependencies

In [1]:
!pip install -q -r ./smolagents/requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: './smolagents/requirements.txt'


## Generation with OpenVINO GenAI
We can simply run generation with the Qwen3-8B model using OpenVINO GenAI library.
[OpenVINO™ GenAI](https://github.com/openvinotoolkit/openvino.genai) is a library of the most popular Generative AI model pipelines, optimized execution methods, and samples that run on top of highly performant OpenVINO Runtime.
This library is friendly to PC and laptop execution, and optimized for resource consumption. It requires no external dependencies to run generative models as it already includes all the core functionality (e.g. tokenization via openvino-tokenizers).

### Prepare Model

First we will download Qwen3-8B model converted to the OpenVINO™ IR (Intermediate Representation) format with weights compressed to INT4 by [NNCF](https://github.com/openvinotoolkit/nncf), available in the OpenVINO LLM collection [Qwen3-8b-int4-ov](https://huggingface.co/OpenVINO/Qwen3-8B-int4-ov ):

In [2]:
from pathlib import Path
import huggingface_hub as hf_hub

model_id = "OpenVINO/Qwen3-8B-int4-ov"
model_path = Path(model_id.split("/")[-1])

if not model_path.exists():
    hf_hub.snapshot_download(model_id, local_dir=model_path)

### Instantiate a pipeline with OpenVINO Generate API

We will use [OpenVINO Generate API](https://github.com/openvinotoolkit/openvino.genai/blob/master/src/README.md) to create pipelines to run an inference with OpenVINO Runtime. 

Firstly we need to create a pipeline with `LLMPipeline`. `LLMPipeline` is the main object used for text generation using LLM in OpenVINO GenAI API. You can construct it straight away from the folder with the downloaded model. We will provide directory with model and device for `LLMPipeline`. Additionally we provide `SchedulerConfig` for resource management.  Then we can run `generate` method and get the output in text format.

In [3]:
import openvino_genai as ov_genai
import time


def streamer(subword):
    print(subword, end="", flush=True)
    return False


# select device for inference
device = "CPU"

# define scheduler
scheduler_config = ov_genai.SchedulerConfig()
scheduler_config.num_kv_blocks = 200
scheduler_config.dynamic_split_fuse = False
scheduler_config.max_num_batched_tokens = 8192
scheduler_config.enable_prefix_caching = False
scheduler_config.use_cache_eviction = False

# create a pipeline for generation
pipe = ov_genai.LLMPipeline(model_path, device, scheduler_config=scheduler_config)

Model: OV Tokenizer
  NETWORK_NAME: tokenizer
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 32
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.4.1;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD EPYC 9334 32-Core Processor                
Model: OV Detokenizer
  NETWORK_NAME: detokenizer


Model: LLM with Paged Attention
  NETWORK_NAME: Model0
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 32
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.4.1;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD EPYC 9334 32-Core Processor                



After instantiating the pipeline we are ready to generate with the model.

We can configure parameters for decoding. We can create the default config with `ov_genai.GenerationConfig()`, setup parameters, and apply the updated version with `set_generation_config(config)` or put config directly to `generate()`. Since our prompt is already formatted in the Qwen3 chat-template format, we set the generation-config 'apply_chat_template' parameter to 'False'.
To get a more accurate measurement of the generation time, we add a warmup generation step before the actual generation to let the model allocate memory and compile any kernels it needs to reach its full potential.

In [4]:
import time

generation_config = ov_genai.GenerationConfig()
generation_config.apply_chat_template = False

input_prompt = """<|im_start|>user
In one sentence, explain what blockchain is.
<|im_end|>
<|im_start|>assistant
"""

# We will first do a short warmup to the model so the time measurement will not include the warmup overhead.
generation_config.max_new_tokens = 100
pipe.generate(input_prompt, generation_config)

# Now we can measure the time and see the result
generation_config.max_new_tokens = 2048

start = time.perf_counter()
result = pipe.generate([input_prompt], generation_config, streamer)
# we don't include TTFT in speedup measurement
ar_gen_time = time.perf_counter() - start - (result.perf_metrics.get_ttft().mean / 1000)

<think>


Okay

,

 the

 user

...

-resistant

 chain

 of

 blocks

.

In [5]:
print(f"Generation took {ar_gen_time:.3f} seconds")

import gc

# del pipe
# gc.collect()

Generation took 49.051 seconds


## Accelerated Generation with Speculative Decoding
Speculative decoding is a lossless decoding paradigm introduced in a recent [ICML paper](https://arxiv.org/abs/2211.17192) for accelerating auto-regressive generation with LLMs.
The method aims to mitigate the inherent latency bottleneck caused by the sequential nature of auto-regressive generation.
Speculative decoding employs a draft language model to generate a block of \(\gamma\) candidate tokens.
The LLM, referred to as the target model, then processes these candidate tokens in parallel.
The algorithm examines each token's probability distribution, calculated by both the target and draft models, to determine whether the token should be accepted or rejected.

In this section we will demonstrate how to accelerate the generation of Qwen3-8B using speculative-decoding, with the open source Qwen3-0.6B model as the draft model.

The [Qwen3‑0.6B](https://huggingface.co/Qwen/Qwen3-0.6B) model is a compact yet capable language model in Alibaba’s Qwen3 series, featuring just 0.6 billion parameters (~0.44B non-embedding), 28 layers, and a 32K token context window, making it ideal for edge or low-resource deployments. Despite its small size, it's built atop the same hybrid reasoning architecture as its larger siblings, supporting both thinking mode (for logical reasoning, math, and code) and non-thinking mode (for fast, conversational responses) within a unified framework.

We will first download the draft-model and then we will use it to initialize a speculative-decoding generation pipeline:

In [6]:
from pathlib import Path
import huggingface_hub as hf_hub

draft_model_id = "OpenVINO/Qwen3-0.6B-int8-ov"
draft_model_path = Path(draft_model_id.split("/")[-1])

if not draft_model_path.exists():
    hf_hub.snapshot_download(draft_model_id, local_dir=draft_model_path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

In [7]:
# Define schedulers for main/draft models

draft_device = "CPU"
scheduler_config = ov_genai.SchedulerConfig()
scheduler_config.num_kv_blocks = 200
scheduler_config.dynamic_split_fuse = False
scheduler_config.max_num_batched_tokens = 8192
scheduler_config.enable_prefix_caching = False
scheduler_config.use_cache_eviction = False

draft_scheduler_config = ov_genai.SchedulerConfig()
draft_scheduler_config.num_kv_blocks = 200
draft_scheduler_config.dynamic_split_fuse = False
draft_scheduler_config.max_num_batched_tokens = 8192

draft_model = ov_genai.draft_model(draft_model_path, draft_device, scheduler_config=draft_scheduler_config)

# create a pipeline with a draft model for generation with speculative decoding
pipe = ov_genai.LLMPipeline(model_path, device, draft_model=draft_model, scheduler_config=scheduler_config)

Model: OV Tokenizer
  NETWORK_NAME: tokenizer
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 32
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.4.1;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD EPYC 9334 32-Core Processor                
Model: OV Detokenizer
  NETWORK_NAME: detokenizer


Model: LLM with Paged Attention
  NETWORK_NAME: Model0
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 32
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.4.1;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD EPYC 9334 32-Core Processor                


Model: LLM with Paged Attention
  NETWORK_NAME: Model0
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 32
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.4.1;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD EPYC 9334 32-Core Processor                


Now we are ready to generate with our speculative decoding pipeline. We will run a small warmup step before measuring the actual generation time

In [8]:
# Define in the generation config the numbers of candidates generated by draft_model per iteration
generation_config.num_assistant_tokens = 3
generation_config.apply_chat_template = False

# Again we will do a short warmup before measuring time for the model
generation_config.max_new_tokens = 100
pipe.generate(input_prompt, generation_config)

# Now we can measure the time and see the result
generation_config.max_new_tokens = 2048

start = time.perf_counter()
result = pipe.generate([input_prompt], generation_config, streamer)
# we don't include TTFT in speedup measurement
sd_gen_time = time.perf_counter() - start - (result.perf_metrics.get_ttft().mean / 1000)


Generated tokens: 100
Total duration, sec: 10.3439
Draft model duration, sec: 3.34305
Main model duration, sec: 7.00083
Draft model duration, %: 32.3191
Main model duration, %: 67.6809
Token per sec: 9.66755
AVG acceptance rate, %: 50.7368
REQUEST_ID: 0
Main model iterations: 38
AVG acceptance rate, %: 50.7368
Generated tokens by draft model: 116
Accepted token rate, %: 50
Request_id: 0 ||| 100 100 100 100 100 0 100 33.3333 33.3333 0 0 66.6667 66.6667 0 100 66.6667 0 0 100 66.6667 0 100 0 33.3333 100 33.3333 66.6667 33.3333 100 0 0 100 0 0 33.3333 100 100 0 


<think>
Okay

, the user wants

 a one-sentence

 explanation of blockchain.

...

 chain of blocks

, ensuring transparency and

 immutability across


Generated tokens: 435
Total duration, sec: 32.3188
Draft model duration, sec: 14.1124
Main model duration, sec: 18.2064
Draft model duration, %: 43.6663
Main model duration, %: 56.3337
Token per sec: 13.4596
AVG acceptance rate, %: 59.5909
REQUEST_ID: 0
Main model iterations: 154
AVG acceptance rate, %: 59.5909
Generated tokens by draft model: 465
Accepted token rate, %: 59.3548
Request_id: 0 ||| 100 100 100 100 100 0 100 33.3333 33.3333 0 0 66.6667 66.6667 0 100 66.6667 0 0 100 66.6667 0 100 0 33.3333 100 33.3333 66.6667 33.3333 100 0 0 100 0 0 33.3333 100 100 0 100 0 66.6667 100 0 100 0 0 0 100 0 0 66.6667 66.6667 100 0 66.6667 100 0 33.3333 33.3333 66.6667 100 100 100 100 100 100 66.6667 66.6667 66.6667 100 66.6667 100 33.3333 100 0 100 0 33.3333 100 100 0 100 66.6667 100 100 100 100 0 100 33.3333 100 100 100 33.3333 100 0 66.6667 100 100 100 0 100 33.3333 0 66.6667 66.6667 100 66.6667 0 0 66.6667 0 33.3333 100 100 0 0 0 100 33.3333 100 66.6667 0 0 100 0 100 100 33.3333 33.3333 33.

.

In [9]:
print(f"Generation took {sd_gen_time:.3f} seconds")

Generation took 32.158 seconds


Let's calculate the speedup achieved when accelerating using Qwen3-0.6B as a draft for the specific example we used:

In [10]:
print(f"End to end speedup with speculative decoding is {ar_gen_time / sd_gen_time:.2f}x")

End to end speedup with speculative decoding is 1.53x


## Further Accelerate Qwen3-8B by Draft Layer Pruning 

Layer pruning is a model compression technique for large language models (LLMs) that reduces inference cost by removing entire transformer layers from the network. We leveraged this technique to generate a smaller yet qualitative draft-model - [pruned-qwen3-draft](https://huggingface.co/OpenVINO/Qwen3-pruned-6L-from-0.6B-int8-ov), eventually further accelerating the speculative-decoding generation of Qwen3-8B model.

We will download the enhanced draft-model and repeat the measurement in previous section to demonstrate the speedup improvement:


In [11]:
from pathlib import Path
import huggingface_hub as hf_hub

draft_model_id = "OpenVINO/Qwen3-pruned-6L-from-0.6B-int8-ov"
draft_model_path = Path(draft_model_id.split("/")[-1])

if not draft_model_path.exists():
    hf_hub.snapshot_download(draft_model_id, local_dir=draft_model_path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

In [12]:
# Define schedulers for main/draft models

draft_device = "CPU"
scheduler_config = ov_genai.SchedulerConfig()
scheduler_config.num_kv_blocks = 200
scheduler_config.dynamic_split_fuse = False
scheduler_config.max_num_batched_tokens = 8192
scheduler_config.enable_prefix_caching = False
scheduler_config.use_cache_eviction = False

draft_scheduler_config = ov_genai.SchedulerConfig()
draft_scheduler_config.num_kv_blocks = 200
draft_scheduler_config.dynamic_split_fuse = False
draft_scheduler_config.max_num_batched_tokens = 8192

draft_model = ov_genai.draft_model(draft_model_path, draft_device, scheduler_config=draft_scheduler_config)

# create a pipeline with a draft model for generation with speculative decoding
pipe = ov_genai.LLMPipeline(model_path, device, draft_model=draft_model, scheduler_config=scheduler_config)

Model: OV Tokenizer
  NETWORK_NAME: tokenizer
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 32
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.4.1;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD EPYC 9334 32-Core Processor                
Model: OV Detokenizer
  NETWORK_NAME: detokenizer


Model: LLM with Paged Attention
  NETWORK_NAME: Model0
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 32
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.4.1;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD EPYC 9334 32-Core Processor                


Model: LLM with Paged Attention
  NETWORK_NAME: Model0
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 32
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.4.1;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD EPYC 9334 32-Core Processor                


In [13]:
# We need to define in the generation config how many tokens the draft should predict in each cycle
generation_config.num_assistant_tokens = 3
generation_config.apply_chat_template = False

# Again we will do a short warmup before measuring time for the model
generation_config.max_new_tokens = 100
pipe.generate(input_prompt, generation_config)

# Now we can measure the time and see the result
generation_config.max_new_tokens = 2048

start = time.perf_counter()
result = pipe.generate([input_prompt], generation_config, streamer)
accelerated_sd_gen_time = time.perf_counter() - start - (result.perf_metrics.get_ttft().mean / 1000)  # we don't include TTFT in speedup measurement


Generated tokens: 100
Total duration, sec: 8.89088
Draft model duration, sec: 2.4447
Main model duration, sec: 6.44618
Draft model duration, %: 27.4968
Main model duration, %: 72.5032
Token per sec: 11.2475
AVG acceptance rate, %: 57.9429
REQUEST_ID: 0
Main model iterations: 35
AVG acceptance rate, %: 57.9429
Generated tokens by draft model: 107
Accepted token rate, %: 57.0093
Request_id: 0 ||| 100 100 100 0 100 100 100 33.3333 33.3333 0 0 66.6667 66.6667 0 33.3333 66.6667 33.3333 100 100 0 0 100 0 100 66.6667 100 100 66.6667 100 33.3333 100 0 100 33.3333 0 


<think>
Okay

, the user wants

 a one-sentence

 explanation

...

-resistant chain of blocks

, ensuring transparency and

 immutability across


Generated tokens: 434
Total duration, sec: 27.7082
Draft model duration, sec: 10.5587
Main model duration, sec: 17.1495
Draft model duration, %: 38.1068
Main model duration, %: 61.8932
Token per sec: 15.6633
AVG acceptance rate, %: 64.4931
REQUEST_ID: 0
Main model iterations: 146
AVG acceptance rate, %: 64.4931
Generated tokens by draft model: 441
Accepted token rate, %: 64.1723
Request_id: 0 ||| 100 100 100 0 100 100 100 33.3333 33.3333 0 0 66.6667 66.6667 0 33.3333 66.6667 33.3333 100 100 0 0 100 0 100 66.6667 100 100 66.6667 100 33.3333 100 0 100 33.3333 0 100 0 100 0 0 100 66.6667 0 100 33.3333 0 66.6667 66.6667 100 0 33.3333 0 0 100 100 0 100 100 66.6667 100 100 0 66.6667 100 100 0 0 100 33.3333 66.6667 100 100 100 100 0 66.6667 100 100 33.3333 66.6667 0 100 100 100 100 100 100 100 0 0 0 100 100 0 100 100 100 0 100 33.3333 100 100 100 100 0 0 33.3333 66.6667 33.3333 66.6667 100 100 100 33.3333 0 0 66.6667 100 66.6667 33.3333 100 100 33.3333 100 100 100 100 100 100 100 0 66.6667 1

.

In [14]:
print(f"Generation took {accelerated_sd_gen_time:.3f} seconds")
print(f"End to end speedup with speculative decoding is {ar_gen_time / accelerated_sd_gen_time:.2f}x")

Generation took 27.586 seconds
End to end speedup with speculative decoding is 1.78x


## Compute Average Speedup Gain of Speculative Decoding with Qwen3-8B model

In this section we measure the average speedup gain of speculative-decoding with Qwen3-8B model over multiple examples. 
We use a small mix of summarization, reasoning, math and classification examples for the speedup evaluation. The examples are taken from the [CNN/DailyMail](https://huggingface.co/datasets/abisee/cnn_dailymail) and the [MT-Bench](https://huggingface.co/datasets/philschmid/mt-bench) datasets.  
We run the model on these examples twice: first without speculative-decoding, then with speculative-decoding. We compute the total time ratio of the two methods- which is the speedup, and finally we compute the average speedup over all tested examples.

### 1. Run target model without speculative decoding
We will first run generation without speculative-decoding, but this time we will run it over multiple examples. The examples are provided in a json file.

In [15]:
import openvino_genai as ov_genai
import sys
import time
from tqdm import tqdm

print(f"Loading model from {model_path}")

# Define scheduler
scheduler_config = ov_genai.SchedulerConfig()
scheduler_config.num_kv_blocks = 200
scheduler_config.dynamic_split_fuse = False
scheduler_config.max_num_batched_tokens = 8192
scheduler_config.enable_prefix_caching = False
scheduler_config.use_cache_eviction = False

pipe = ov_genai.LLMPipeline(model_path, device, scheduler_config=scheduler_config)

generation_config = ov_genai.GenerationConfig()
generation_config.apply_chat_template = False

print("Loading prompts...")
import json

f = open("prompts.json")
prompts = json.load(f)

# We will first do a short warmup to the model so the time measurement will not include the warmup overhead.
generation_config.max_new_tokens = 100
pipe.generate("This is a warmup prompt", generation_config)

# finished warmup step, let's run our examples
generation_config.max_new_tokens = 2048
times_auto_regressive = []
print("Running Auto-Regressive generation...")
for prompt in tqdm(prompts):
    start_time = time.perf_counter()
    result = pipe.generate(prompt, generation_config)
    end_time = time.perf_counter()
    times_auto_regressive.append(end_time - start_time)
print("Done")

import gc

del pipe
gc.collect()

Loading model from Qwen3-8B-int4-ov
Model: OV Tokenizer
  NETWORK_NAME: tokenizer
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 32
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.4.1;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD EPYC 9334 32-Core Processor                
Model: OV Deto

Model: LLM with Paged Attention
  NETWORK_NAME: Model0
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 32
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.4.1;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD EPYC 9334 32-Core Processor                


Loading prompts...


Running Auto-Regressive generation...


  0%|          | 0/21 [00:00<?, ?it/s]

...

 95%|█████████▌| 20/21 [26:57<01:36, 96.52s/it] 

100%|██████████| 21/21 [27:53<00:00, 84.41s/it]

100%|██████████| 21/21 [27:53<00:00, 79.71s/it]

Done


120

### 2. Run target model with speculative decoding
Now we will run generation with speculative-decoding over the same examples. 

In the following dropdown list you can choose which draft to use for the speculative-decoding pipeline:

In [16]:
import ipywidgets as widgets

draft_model_name = widgets.Dropdown(
    options=["Qwen3-0.6B-int8-ov", "Qwen3-0.6B-pruned-int8-ov"],
    value="Qwen3-0.6B-int8-ov",  # default value
    description="Select Draft Model:",
)

draft_model_name

Dropdown(description='Select Draft Model:', options=('Qwen3-0.6B-int8-ov', 'Qwen3-0.6B-pruned-int8-ov'), value…

In [17]:
import openvino_genai as ov_genai
import time
from tqdm import tqdm

draft_model_path = draft_model_name.value

print("Loading prompts...")
import json

f = open("prompts.json")
prompts = json.load(f)

# Define scheduler
scheduler_config = ov_genai.SchedulerConfig()
scheduler_config.num_kv_blocks = 200
scheduler_config.dynamic_split_fuse = False
scheduler_config.max_num_batched_tokens = 8192
scheduler_config.enable_prefix_caching = False
scheduler_config.use_cache_eviction = False
# Define scheduler for the draft

draft_scheduler_config = ov_genai.SchedulerConfig()
draft_scheduler_config.num_kv_blocks = 200
draft_scheduler_config.dynamic_split_fuse = False
draft_scheduler_config.max_num_batched_tokens = 8192

draft_model = ov_genai.draft_model(draft_model_path, device, scheduler_config=draft_scheduler_config)

pipe = ov_genai.LLMPipeline(model_path, device, draft_model=draft_model, scheduler_config=scheduler_config)

generation_config = ov_genai.GenerationConfig()
generation_config.num_assistant_tokens = 3
generation_config.apply_chat_template = False

# Again, We will first do a short warmup
generation_config.max_new_tokens = 100
pipe.generate("This is a warmup prompt", generation_config)

# finished warmup step, let's run our examples
generation_config.max_new_tokens = 2048
times_speculative_decoding = []

print("Running Speculative Decoding generation...")
for prompt in tqdm(prompts):
    start_time = time.perf_counter()
    result = pipe.generate(prompt, generation_config)
    end_time = time.perf_counter()
    times_speculative_decoding.append(end_time - start_time)
print("Done")

Loading prompts...
Model: OV Tokenizer
  NETWORK_NAME: 

tokenizer
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 32
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.4.1;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD EPYC 9334 32-Core Processor                
Model: OV Detokenizer
  NETWORK_NAME: detokenizer
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 

1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 32
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.4.1;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD EPYC 9334 32-Core Processor                
Model: OV Tokenizer
  NETWORK_NAME: tokenizer
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 32
  P

Model: LLM with Paged Attention
  NETWORK_NAME: Model0
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 32
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.4.1;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD EPYC 9334 32-Core Processor                


Model: LLM with Paged Attention
  NETWORK_NAME: Model0
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: 32
  PERF_COUNT: NO
  INFERENCE_PRECISION_HINT: bf16
  PERFORMANCE_HINT: LATENCY
  EXECUTION_MODE_HINT: PERFORMANCE
  PERFORMANCE_HINT_NUM_REQUESTS: 0
  ENABLE_CPU_PINNING: YES
  ENABLE_CPU_RESERVATION: NO
  SCHEDULING_CORE_TYPE: ANY_CORE
  MODEL_DISTRIBUTION_POLICY: 
  ENABLE_HYPER_THREADING: NO
  EXECUTION_DEVICES: CPU
  CPU_DENORMALS_OPTIMIZATION: NO
  LOG_LEVEL: LOG_NONE
  CPU_SPARSE_WEIGHTS_DECOMPRESSION_RATE: 1
  ENABLE_TENSOR_PARALLEL: NO
  TBB_PARTITIONER: STATIC
  DYNAMIC_QUANTIZATION_GROUP_SIZE: 32
  KV_CACHE_PRECISION: u8
  KEY_CACHE_PRECISION: u8
  VALUE_CACHE_PRECISION: u8
  KEY_CACHE_GROUP_SIZE: 0
  VALUE_CACHE_GROUP_SIZE: 0
  RUNTIME_REQUIREMENTS: meta=1.0;ov=2026.4.1;isa=Intel AVX-512 with Intel DL Boost and bfloat16 support
EXECUTION_DEVICES:
 CPU: AMD EPYC 9334 32-Core Processor                


...

 95%|█████████▌| 20/21 [20:01<01:06, 66.15s/it]


Generated tokens: 535
Total duration, sec: 36.8318
Draft model duration, sec: 15.5616
Main model duration, sec: 21.2701
Draft model duration, %: 42.2506
Main model duration, %: 57.7494
Token per sec: 14.5255
AVG acceptance rate, %: 63.2131
REQUEST_ID: 0
Main model iterations: 183
AVG acceptance rate, %: 63.2131
Generated tokens by draft model: 552
Accepted token rate, %: 63.0435
Request_id: 0 ||| 100 100 66.6667 0 33.3333 100 33.3333 66.6667 100 100 0 100 100 100 100 33.3333 33.3333 66.6667 0 0 0 33.3333 100 100 66.6667 33.3333 66.6667 66.6667 100 100 100 66.6667 100 100 100 100 0 66.6667 33.3333 0 100 0 66.6667 66.6667 33.3333 33.3333 33.3333 66.6667 0 100 33.3333 100 100 100 100 100 66.6667 100 33.3333 100 33.3333 100 100 100 66.6667 100 100 66.6667 100 0 0 100 100 0 66.6667 66.6667 0 0 0 100 0 33.3333 33.3333 100 0 100 100 66.6667 0 0 33.3333 100 33.3333 66.6667 100 100 33.3333 100 0 66.6667 100 33.3333 100 100 0 100 100 66.6667 0 66.6667 0 100 100 33.3333 0 0 33.3333 33.3333 33.33

100%|██████████| 21/21 [20:37<00:00, 57.13s/it]

100%|██████████| 21/21 [20:37<00:00, 58.93s/it]


Generated tokens: 511
Total duration, sec: 36.0933
Draft model duration, sec: 15.2998
Main model duration, sec: 20.7935
Draft model duration, %: 42.3895
Main model duration, %: 57.6105
Token per sec: 14.1578
AVG acceptance rate, %: 57.6703
REQUEST_ID: 0
Main model iterations: 185
AVG acceptance rate, %: 57.6703
Generated tokens by draft model: 557
Accepted token rate, %: 57.6302
Request_id: 0 ||| 100 0 100 66.6667 66.6667 100 100 100 100 100 66.6667 100 0 100 66.6667 33.3333 0 66.6667 0 100 33.3333 66.6667 0 0 100 100 33.3333 66.6667 0 100 66.6667 0 33.3333 0 0 0 0 0 66.6667 100 100 100 100 100 66.6667 0 0 100 0 0 33.3333 0 33.3333 0 33.3333 0 33.3333 100 33.3333 33.3333 100 100 66.6667 100 100 0 0 0 100 0 66.6667 0 100 33.3333 66.6667 66.6667 33.3333 0 0 33.3333 33.3333 33.3333 100 100 0 0 33.3333 100 100 33.3333 66.6667 66.6667 33.3333 0 100 66.6667 100 33.3333 66.6667 100 0 0 33.3333 0 0 33.3333 100 100 100 33.3333 0 66.6667 100 0 0 66.6667 33.3333 66.6667 33.3333 66.6667 66.6667 0

### 3. Calculate average speedup


In [18]:
avg_speedup = sum([x / y for x, y in zip(times_auto_regressive, times_speculative_decoding)]) / len(prompts)
print(f"average speedup: {avg_speedup:.2f}")

average speedup: 1.33
